# 🏈 Sports Betting Odds & GCP BigQuery Exploration

This notebook allows interactive exploration of:
1. **The Odds API**: Live endpoint requests, quota/usage tracking (`x-requests-remaining`), and FanDuel line parsing for NFL & NCAAF.
2. **Data Normalization**: Flattening raw nested API responses into the clean 10-column snapshot schema.
3. **Free Tier Quota Simulator**: Planning scheduling intervals to stay within the 500 requests/month free tier.
4. **Google Cloud BigQuery**: Table schema inspection, dry-run query cost estimation, live table queries, and test inserts.
5. **Interactive Visualizations**: Visualizing spread, total (O/U), and moneyline movements over time with Plotly.

## 1. Setup & Environment Diagnostics
Load configuration from `.env` and initialize GCP and API credentials.

In [ ]:
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import plotly.express as px

# Ensure repository root is on sys.path
ROOT = Path.cwd().resolve()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app.config.config import my_api_key, GCP_PROJECT_ID, DATASET_ID, gcp_service_accnt
from app.ddl.table_schemas import odds_schema
from app.query_betting_api import OddsQuery

print("=== Environment Diagnostics ===")
print(f"Project Root:       {ROOT}")
print(f"The Odds API Key:   {'[LOADED] ' + my_api_key[:6] + '...' if my_api_key else '[MISSING - Set in .env]'}")
print(f"GCP Project ID:     {GCP_PROJECT_ID}")
print(f"BigQuery Dataset:   {DATASET_ID}")
print(f"GCP Credentials:    {'[LOADED] ' + str(type(gcp_service_accnt).__name__) if gcp_service_accnt else '[NOT FOUND - Run gcloud auth application-default login]'}")

## 2. Free Tier Quota Calculator
The Odds API free plan offers **500 requests per month**.
- Querying 1 sport (e.g. NFL) with all markets (`h2h,spreads,totals`) costs **1 request**.
- Querying both NFL and NCAA Football costs **2 requests per snapshot**.

Run this calculator to see how different ingestion schedules compare against your monthly budget.

In [ ]:
def calculate_quota_usage(sports_count=2, monthly_limit=500):
    schedules = {
        "24/7 Hourly (Naive)": 24 * 30 * sports_count,
        "24/7 Every 2 Hours": 12 * 30 * sports_count,
        "24/7 Every 3 Hours": 8 * 30 * sports_count,
        "Game Windows Hourly (Thu-Mon active hours)": (8 + 7 + 14 + 14 + 7 + 4) * 4.2 * sports_count,
        "Twice Daily (12:00 & 18:00 UTC)": 2 * 30 * sports_count,
    }
    
    rows = []
    for name, monthly_reqs in schedules.items():
        reqs = int(round(monthly_reqs))
        status = "✅ Within Free Tier" if reqs <= monthly_limit else f"❌ Over Limit by {reqs - monthly_limit}"
        rows.append({
            "Schedule": name,
            "Est. Monthly Requests": reqs,
            "% of Free Quota": f"{(reqs / monthly_limit) * 100:.1f}%",
            "Status": status
        })
    return pd.DataFrame(rows)

df_quota = calculate_quota_usage(sports_count=2, monthly_limit=500)
df_quota

## 3. The Odds API: Live Calls, Quota Tracking & Raw Payloads
Test live API requests, inspect response headers (`x-requests-remaining`), and view the raw JSON payload.

In [ ]:
API_KEY = my_api_key or os.getenv("THE_ODDS_API_KEY")
BASE_URL = "https://api.the-odds-api.com"
LEAGUES = ["americanfootball_nfl", "americanfootball_ncaaf"]

def fetch_odds_live(sport_key="americanfootball_nfl", markets="h2h,spreads,totals", regions="us"):
    if not API_KEY:
        print("⚠️ No API key found. Using local fixture sample.")
        return None, {}
    
    url = f"{BASE_URL}/v4/sports/{sport_key}/odds"
    params = {
        "apiKey": API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": "american",
    }
    
    res = requests.get(url, params=params, timeout=15)
    quota_headers = {
        "status_code": res.status_code,
        "requests_remaining": res.headers.get("x-requests-remaining"),
        "requests_used": res.headers.get("x-requests-used"),
        "requests_last": res.headers.get("x-requests-last"),
    }
    
    if res.status_code == 200:
        return res.json(), quota_headers
    else:
        print(f"❌ Request failed [{res.status_code}]: {res.text}")
        return None, quota_headers

# Test request for NFL
raw_events, headers = fetch_odds_live("americanfootball_nfl")
print("API Response Status:", headers)
if raw_events:
    print(f"Successfully fetched {len(raw_events)} events.")
else:
    print("Loading fallback fixture from test suite...")
    from app.tests.test_the_odds_api_parsing import SAMPLE_EVENT
    raw_events = [SAMPLE_EVENT]
    print(f"Loaded {len(raw_events)} fixture event(s).")

### Inspecting the Raw Event Structure
Review how FanDuel formats bookmakers, markets, and outcomes.

In [ ]:
sample_event = raw_events[0] if raw_events else {}
print(f"Game: {sample_event.get('away_team')} @ {sample_event.get('home_team')}")
print(f"Commence Time: {sample_event.get('commence_time')}")

# Find FanDuel bookmaker entry
fanduel = next((b for b in sample_event.get("bookmakers", []) if b.get("key").lower() == "fanduel"), None)
if fanduel:
    print("\nFanDuel Markets Available:", [m.get("key") for m in fanduel.get("markets", [])])
    print("\nSample Market Data (JSON):")
    print(json.dumps(fanduel.get("markets", [])[:2], indent=2))
else:
    print("\nFanDuel line not yet posted for this matchup.")
    all_books = [b.get("key") for b in sample_event.get("bookmakers", [])]
    print("Other bookmakers available in response:", all_books[:8])

## 4. Normalization: Converting to the 10-Column Snapshot Table
Flattens raw events into our standardized BigQuery schema.

In [ ]:
def normalize_events(events, target_bookmaker="fanduel"):
    rows = []
    snapshot_time = datetime.now(timezone.utc).isoformat()
    
    for event in events or []:
        if not isinstance(event, dict):
            continue
        game_id = event.get("id")
        sport = event.get("sport_key") or event.get("sport") or "americanfootball_nfl"
        away_team = event.get("away_team")
        home_team = event.get("home_team")
        commence_time = event.get("commence_time")
        
        for bookmaker in event.get("bookmakers", []):
            if str(bookmaker.get("key", "")).lower() != target_bookmaker.lower():
                continue
            
            for market in bookmaker.get("markets", []):
                market_type = market.get("key")  # h2h, spreads, totals
                for outcome in market.get("outcomes", []):
                    try:
                        price = int(outcome.get("price"))
                    except (TypeError, ValueError):
                        price = 0
                    try:
                        market_point = float(outcome.get("point")) if outcome.get("point") is not None else None
                    except (TypeError, ValueError):
                        market_point = None
                        
                    rows.append({
                        "snapshot_time": snapshot_time,
                        "game_id": str(game_id),
                        "sport": sport,
                        "away_team": away_team,
                        "home_team": home_team,
                        "commence_time": commence_time,
                        "market_type": market_type,
                        "team_or_outcome": outcome.get("name"),
                        "price": price,
                        "market_point": market_point
                    })
    
    cols = [
        "snapshot_time", "game_id", "sport", "away_team", "home_team",
        "commence_time", "market_type", "team_or_outcome", "price", "market_point"
    ]
    return pd.DataFrame(rows, columns=cols)

df_snapshot = normalize_events(raw_events)
print(f"Normalized Rows: {len(df_snapshot)}")
df_snapshot.head(10)

### Profiling Snapshot Data

In [ ]:
if not df_snapshot.empty:
    print("=== Market Counts ===")
    print(df_snapshot["market_type"].value_counts())
    print("\n=== Data Types ===")
    print(df_snapshot.dtypes)
    print("\n=== Null Values ===")
    print(df_snapshot.isna().sum())
else:
    print("No rows normalized. Check if FanDuel is included in bookmakers.")

## 5. Google Cloud BigQuery Exploration
Connect to BigQuery, list datasets/tables, inspect schema, and estimate query costs.

In [ ]:
from google.cloud import bigquery

try:
    bq_client = bigquery.Client(project=GCP_PROJECT_ID, credentials=gcp_service_accnt)
    datasets = list(bq_client.list_datasets())
    print(f"✅ Successfully connected to BigQuery project: '{GCP_PROJECT_ID}'")
    print(f"Available Datasets ({len(datasets)}): {[d.dataset_id for d in datasets]}")
    
    # List tables in target dataset
    tables = list(bq_client.list_tables(DATASET_ID))
    print(f"Tables in '{DATASET_ID}' ({len(tables)}): {[t.table_id for t in tables]}")
except Exception as e:
    print(f"⚠️ BigQuery connection unavailable: {e}")
    print("Run 'gcloud auth application-default login' in terminal to authenticate.")
    bq_client = None

### BigQuery Table DDL & Schema Verification
Generate the `CREATE TABLE` DDL statement to initialize `daily_odds_data` / `fanduel_history`.

In [ ]:
target_table_id = f"{GCP_PROJECT_ID}.{DATASET_ID}.daily_odds_data"

def generate_ddl(table_id, schema):
    fields = ",\n".join(f"  `{f.name}` {f.field_type}" for f in schema)
    return f"CREATE TABLE IF NOT EXISTS `{table_id}` (\n{fields}\n);"

ddl_sql = generate_ddl(target_table_id, odds_schema)
print(ddl_sql)

# Uncomment below if you want to create the table directly from the notebook:
# if bq_client:
#     table = bigquery.Table(target_table_id, schema=odds_schema)
#     created = bq_client.create_table(table, exists_ok=True)
#     print(f"✅ Table ready: {created.full_table_id}")

### BigQuery Dry-Run Query Cost Estimator
Use `dry_run=True` to calculate the exact bytes processed by any query before spending money.

In [ ]:
if bq_client:
    sample_query = f"""
        SELECT snapshot_time, game_id, sport, away_team, home_team, market_type, price, market_point
        FROM `{target_table_id}`
        ORDER BY snapshot_time DESC
        LIMIT 100
    """
    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    try:
        dry_run_job = bq_client.query(sample_query, job_config=job_config)
        bytes_mb = dry_run_job.total_bytes_processed / (1024 * 1024)
        print(f"Dry Run Successful!")
        print(f"Estimated Data Processed: {bytes_mb:.2f} MB")
        print(f"Estimated Cost:           ${bytes_mb / 1024 / 1024 * 5:.6f} (Free tier covers 1TB/month)")
    except Exception as e:
        print(f"Dry Run Note: {e}")
else:
    print("BigQuery client not connected.")

## 6. Interactive Visualizations: Plotly Line Movement Trends
Visualize how FanDuel spreads, totals, and moneylines fluctuate over time for a matchup.

In [ ]:
# Synthetic movement generator (used for demonstration if historical table is empty)
def create_demo_line_movement():
    timestamps = ["2026-09-01T12:00:00Z", "2026-09-02T12:00:00Z", "2026-09-03T12:00:00Z", "2026-09-04T12:00:00Z"]
    records = [
        # Spreads movement (Bills moved from -2.5 to -3.5)
        {"snapshot_time": timestamps[0], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Buffalo Bills", "price": -110, "market_point": -2.5},
        {"snapshot_time": timestamps[1], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Buffalo Bills", "price": -115, "market_point": -2.5},
        {"snapshot_time": timestamps[2], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Buffalo Bills", "price": -110, "market_point": -3.0},
        {"snapshot_time": timestamps[3], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Buffalo Bills", "price": -105, "market_point": -3.5},
        # KC Spreads
        {"snapshot_time": timestamps[0], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Kansas City Chiefs", "price": -110, "market_point": 2.5},
        {"snapshot_time": timestamps[1], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Kansas City Chiefs", "price": -105, "market_point": 2.5},
        {"snapshot_time": timestamps[2], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Kansas City Chiefs", "price": -110, "market_point": 3.0},
        {"snapshot_time": timestamps[3], "game_id": "demo_bills_kc", "market_type": "spreads", "team_or_outcome": "Kansas City Chiefs", "price": -115, "market_point": 3.5},
        # Totals movement (Total dropped from 48.5 to 46.5)
        {"snapshot_time": timestamps[0], "game_id": "demo_bills_kc", "market_type": "totals", "team_or_outcome": "Over", "price": -110, "market_point": 48.5},
        {"snapshot_time": timestamps[1], "game_id": "demo_bills_kc", "market_type": "totals", "team_or_outcome": "Over", "price": -112, "market_point": 48.5},
        {"snapshot_time": timestamps[2], "game_id": "demo_bills_kc", "market_type": "totals", "team_or_outcome": "Over", "price": -110, "market_point": 47.5},
        {"snapshot_time": timestamps[3], "game_id": "demo_bills_kc", "market_type": "totals", "team_or_outcome": "Over", "price": -108, "market_point": 46.5},
        # Moneyline movement
        {"snapshot_time": timestamps[0], "game_id": "demo_bills_kc", "market_type": "h2h", "team_or_outcome": "Buffalo Bills", "price": -130, "market_point": None},
        {"snapshot_time": timestamps[1], "game_id": "demo_bills_kc", "market_type": "h2h", "team_or_outcome": "Buffalo Bills", "price": -140, "market_point": None},
        {"snapshot_time": timestamps[2], "game_id": "demo_bills_kc", "market_type": "h2h", "team_or_outcome": "Buffalo Bills", "price": -155, "market_point": None},
        {"snapshot_time": timestamps[3], "game_id": "demo_bills_kc", "market_type": "h2h", "team_or_outcome": "Buffalo Bills", "price": -170, "market_point": None},
    ]
    return pd.DataFrame(records)

df_plot = create_demo_line_movement()

# 1. Plot Spread Movement
spread_data = df_plot[df_plot["market_type"] == "spreads"]
fig_spread = px.line(
    spread_data, 
    x="snapshot_time", 
    y="market_point", 
    color="team_or_outcome",
    markers=True,
    title="FanDuel Spread Line Movement Over Time",
    labels={"market_point": "Point Spread", "snapshot_time": "Snapshot (UTC)"}
)
fig_spread.show()

In [ ]:
# 2. Plot Over / Under Total Movement
totals_data = df_plot[df_plot["market_type"] == "totals"]
fig_totals = px.line(
    totals_data, 
    x="snapshot_time", 
    y="market_point", 
    color="team_or_outcome",
    markers=True,
    title="FanDuel Over/Under Total Line Movement Over Time",
    labels={"market_point": "Total Points Line", "snapshot_time": "Snapshot (UTC)"}
)
fig_totals.show()

In [ ]:
# 3. Plot Moneyline Price Fluctuation (American Odds)
ml_data = df_plot[df_plot["market_type"] == "h2h"]
fig_ml = px.line(
    ml_data, 
    x="snapshot_time", 
    y="price", 
    color="team_or_outcome",
    markers=True,
    title="FanDuel Moneyline Odds Movement (+/- American Odds)",
    labels={"price": "American Odds Price", "snapshot_time": "Snapshot (UTC)"}
)
fig_ml.show()

## 7. Next Steps & Action Checklist
- Check remaining quota via the response headers in Section 3.
- If testing live insertion into BigQuery, run `scripts/preview_bigquery_tables.py --table odds --create`.
- Review the full architecture roadmap at `repo_updates/phase1_next_steps_roadmap.md`.